# MEL — AGENTIC MAX (enfant du checkpoint UNCENSORED)
Cette phase ne modifie jamais le checkpoint UNCENSORED. Elle crée un nouvel adaptateur enfant à partir de trois corpus de tool-use/recherche et des leçons MEL.

In [ ]:
AGENTIC_READY = False  # passer à True uniquement quand Mode complet > LoRA affiche AGENTIC_READY
assert AGENTIC_READY, 'Gate fermé : le checkpoint UNCENSORED doit d abord franchir le benchmark général + impact.'


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU CUDA requis : active un GPU Colab gratuit.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/MEL')
UNCENSORED = ROOT/'lora-uncensored-max'/'uncensored'
AGENTIC_ROOT = ROOT/'lora-agentic-max'
DATA = AGENTIC_ROOT/'data'
OUT = AGENTIC_ROOT/'agentic'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
for name in ['adapter_model.safetensors','adapter_config.json','artifact-evidence.json']:
    assert (UNCENSORED/name).is_file(), f'Checkpoint UNCENSORED manquant: {name}'
print('Parent UNCENSORED:', UNCENSORED)
print('Sortie AGENTIC:', OUT)


In [ ]:
!rm -rf /content/meliturgos-cloudflare
!git clone --branch candidate/mel-clean-autonomy --single-branch https://github.com/adrienlopezcarreras-pixel/meliturgos-cloudflare.git /content/meliturgos-cloudflare
%cd /content/meliturgos-cloudflare
!python -m pip install -U pip
!python -m pip install -r requirements-lora.txt datasets huggingface_hub


## 1 — Corpus AGENTIC massif
xLAM function-calling, ToolACE et DR-TULU deep-research sont exportés dans leur ordre de dataset. Les textes ne sont pas reformulés par le builder.

In [ ]:
from datasets import load_dataset
def export_dataset(repo_id, target):
    target=Path(target)
    if target.exists() and target.stat().st_size>0:
        print('Réutilisé:', target)
        return target
    ds=load_dataset(repo_id, split='train')
    ds.to_json(str(target), orient='records', lines=True, force_ascii=False)
    print(repo_id, '->', len(ds), 'lignes')
    return target

xlam=export_dataset('minpeter/xlam-function-calling-60k-parsed', DATA/'xlam.jsonl')
toolace=export_dataset('AmanPriyanshu/tool-reasoning-sft-TOOLS-toolace-sft-tool-use-agent-data-cleaned-rectified', DATA/'toolace.jsonl')
deep=export_dataset('SupritiVijay/tool-reasoning-sft-RESEARCH-dr-tulu-sft-deep-research-agent-data-cleaned-rectified', DATA/'deep-research.jsonl')
mel=DATA/'mel-lessons.jsonl'
!node scripts/export-canonical-lora-dataset.mjs --output "$mel"


In [ ]:
agentic_data=DATA/'mel-agentic-max.jsonl'
agentic_q=DATA/'mel-agentic-max.quarantine.jsonl'
!python scripts/prepare-mel-agentic-lora.py \
  --xlam "$xlam" \
  --toolace "$toolace" \
  --deep-research "$deep" \
  --mel-lessons "$mel" \
  --output "$agentic_data" \
  --quarantine-output "$agentic_q"
import json
meta=json.loads(Path(str(agentic_data)+'.meta.json').read_text())
print(json.dumps({'examples':meta['examples'],'quarantined':meta['quarantined_examples'],'sha256':meta['output_sha256']},indent=2))


## 2 — Nouveau plan + lignée parent→enfant


In [ ]:
plan=DATA/'lora-plan-agentic-max.json'
!node scripts/create-lora-plan.mjs --dataset "$agentic_data" --output "$plan" --epochs 1 --seed 42
parent=json.loads((UNCENSORED/'artifact-evidence.json').read_text())
parent_digest=parent['digest']
print('Parent artifact digest:', parent_digest)
checkpoints=[]
for p in OUT.glob('checkpoint-*'):
    try: checkpoints.append((int(p.name.split('-')[-1]), p))
    except: pass
resume=max(checkpoints)[1] if checkpoints else None
print('Reprise:', resume or 'nouvel entraînement AGENTIC')


In [ ]:
import subprocess,sys
cmd=[sys.executable,'scripts/train-mel-lora.py',
  '--dataset',str(agentic_data),'--plan',str(plan),'--output',str(OUT),
  '--stage','agentic','--parent-adapter-dir',str(UNCENSORED),'--parent-artifact-digest',parent_digest,
  '--epochs','1','--save-steps','100','--gradient-accumulation-steps','8','--seed','42']
if resume: cmd += ['--resume-from-checkpoint',str(resume)]
subprocess.run(cmd,check=True)


## 3 — Vérification : UNCENSORED intact, AGENTIC séparé


In [ ]:
ue=json.loads((UNCENSORED/'artifact-evidence.json').read_text())
ae=json.loads((OUT/'artifact-evidence.json').read_text())
te=json.loads((OUT/'training-evidence.json').read_text())
assert te['stage']=='agentic'
assert te['parent_artifact_digest']==ue['digest']==parent_digest
assert ae['parent_artifact_digest']==parent_digest
assert ae['digest'] != parent_digest, 'Le checkpoint enfant doit être un artefact distinct.'
import shutil
shutil.copy2(Path(str(agentic_data)+'.meta.json'), OUT/'dataset-metadata.json')
print(json.dumps({'parent_uncensored':parent_digest,'agentic_artifact':ae['digest'],'examples':te['dataset']['examples'],'loss':te['training_metrics']['train_loss']},indent=2))


## 4 — Publication séparée
Le checkpoint AGENTIC est publié dans un dépôt distinct. UNCENSORED reste inchangé et peut toujours être rollbacké.

In [ ]:
from huggingface_hub import HfApi, notebook_login
try:
    me=HfApi().whoami(); print('Hugging Face connecté:',me.get('name') or me.get('fullname'))
except Exception:
    notebook_login()
!python scripts/publish-lora-hf.py --dir "$OUT" --repo-id Meliturgos/mel-lora-agentic
print('AGENTIC publié : https://huggingface.co/Meliturgos/mel-lora-agentic')
